## GPU-Accelerated Notebook
This notebook uses **CuPy** as a drop-in GPU replacement for NumPy,
and **JAX** configured to run on GPU for JIT-compiled Schatten norms.

On Kaggle: go to *Settings → Accelerator → GPU T4 x2* before running.

In [ ]:
# ── Verify GPU is available ──────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip() if result.returncode == 0 else 'Not available – switch accelerator to GPU')


In [ ]:
import numpy as np               # kept for matplotlib & fallback
import matplotlib.pyplot as plt
import torch

# CuPy – GPU NumPy drop-in
try:
    import cupy as cp
    cp.cuda.Device(0).use()          # activate GPU 0
    xp = cp                          # xp = GPU array module
    print('CuPy', cp.__version__, '– using GPU:', cp.cuda.runtime.getDeviceProperties(0)['name'].decode())
except ImportError:
    import numpy as cp               # CPU fallback – silent
    xp = cp
    print('CuPy not found – falling back to NumPy (CPU)')

# JAX – configure GPU backend BEFORE importing jax.numpy
import os
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')  # avoid OOM on shared GPU
try:
    import jax
    import jax.numpy as jnp
    from jax import jit
    JAX_AVAILABLE = True
    print('JAX', jax.__version__, '– default backend:', jax.default_backend())
except ImportError:
    JAX_AVAILABLE = False
    print('JAX not found')


In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch device:', DEVICE)

def torch_schatten_norm(A, p):
    """Schatten p-norm via PyTorch (runs on GPU if available)."""
    A = A.to(DEVICE)
    singular_values = torch.linalg.svdvals(A)
    if p == float('inf'):
        return torch.max(singular_values)
    else:
        return torch.linalg.vector_norm(singular_values, ord=p)


In [ ]:
list_p = [1, 2, 3, 4, 5, 6, float('inf')]
for i in range(1):
    B = torch.randn(5000, 5000)          # will be moved to GPU inside the function
    for p in list_p:
        print('PyTorch Schatten: ', torch_schatten_norm(B, p), p)


In [ ]:
import time
from sklearn.utils.extmath import randomized_svd

# ── 1. NumPy Exact (CPU – kept as reference) ─────────────────────────────────
def schatten_norm_numpy(A, p):
    """Exact Schatten p-norm via NumPy SVD (CPU)."""
    A_np = cp.asnumpy(A) if hasattr(A, 'get') else np.asarray(A)
    s = np.linalg.svd(A_np, compute_uv=False)
    return np.linalg.norm(s, ord=p)

# ── 2. Randomised SVD (CPU via sklearn) ──────────────────────────────────────
def schatten_norm_randomized(A, p, rank=250, n_iter=2):
    """Approximate Schatten p-norm via Randomised SVD."""
    A_np = cp.asnumpy(A) if hasattr(A, 'get') else np.asarray(A)
    max_rank = min(A_np.shape)
    rank = min(rank, max_rank)
    _, s, _ = randomized_svd(A_np, n_components=rank, n_oversamples=100, n_iter=n_iter, random_state=42)
    return np.linalg.norm(s, ord=p)

# ── 3. JAX JIT (GPU via XLA) ─────────────────────────────────────────────────
if JAX_AVAILABLE:
    # JIT only the SVD – ord=p cannot be a traced value, so we keep the
    # norm computation outside JIT as a plain NumPy call on the returned
    # singular values (a tiny 1-D array – negligible cost on CPU).
    @jit
    def _jax_svd(A):
        """JIT-compiled SVD; returns singular values on the XLA device."""
        return jnp.linalg.svd(A, compute_uv=False)

    def schatten_norm_jax(A, p):
        """Exact Schatten p-norm via JAX XLA (GPU if available)."""
        A_np = cp.asnumpy(A) if hasattr(A, 'get') else np.asarray(A)
        s = np.asarray(_jax_svd(jnp.array(A_np)))  # singular values back on CPU
        if p == float('inf'):
            return float(s[0])          # largest singular value
        return float(np.linalg.norm(s, ord=p))
else:
    def schatten_norm_jax(A, p):
        return schatten_norm_numpy(A, p)   # graceful CPU fallback

# ── Benchmark ────────────────────────────────────────────────────────────────
if __name__ == '__main__':
    MATRIX_SIZE = 500
    P_VALUE = 1.2
    print(f'Generating a random {MATRIX_SIZE}x{MATRIX_SIZE} matrix...')
    U, _ = np.linalg.qr(np.random.randn(MATRIX_SIZE, MATRIX_SIZE))
    V, _ = np.linalg.qr(np.random.randn(MATRIX_SIZE, MATRIX_SIZE))
    S = np.diag(1.0 / (np.arange(1, MATRIX_SIZE + 1) ** 0.5))
    A = U @ S @ V.T

    print(f'\n--- Benchmark (p = {P_VALUE}) ---')
    start = time.time()
    res_np = schatten_norm_numpy(A, P_VALUE)
    print(f'[NumPy Exact]      {res_np:.4f} | {time.time()-start:.4f}s')

    start = time.time()
    res_rand = schatten_norm_randomized(A, P_VALUE)
    acc = res_rand / res_np * 100
    print(f'[Randomised SVD]   {res_rand:.4f} | {time.time()-start:.4f}s ({acc:.2f}% accurate)')

    if JAX_AVAILABLE:
        _ = schatten_norm_jax(A, P_VALUE)   # warm-up compilation
        start = time.time()
        res_jax = schatten_norm_jax(A, P_VALUE)
        print(f'[JAX JIT]          {res_jax:.4f} | {time.time()-start:.4f}s')
    else:
        print('[JAX JIT]          Skipped (JAX not found)')


In [ ]:
def relu(A):
    """Element-wise ReLU – works with both NumPy and CuPy arrays."""
    return xp.maximum(0, A)   # vectorised, no Python loop, runs on GPU


In [ ]:
def loss(W, A, Y, n):
    return 1 / (2 * n) * (xp.linalg.norm(W @ A - Y, 'fro') ** 2)

def p1(p):
    if p == xp.inf or p == float('inf'):
        return 1
    return p / (p - 1)

def p2(p):
    if p == 2:
        return float('inf')
    if p == xp.inf or p == float('inf'):
        return 2
    return 2 * p / (p - 2)


In [ ]:
m, n = 100, 1000
W_True = xp.random.randn(m, n)
W_1    = xp.random.randn(n, 50)
X      = xp.random.randn(50, m)
A      = relu(W_1 @ X)
Y      = W_True @ A
W0     = xp.zeros((m, n))


In [ ]:
def find_max(gradient, A, list_p):
    max_norm = 0
    p_max = 0
    for p in list_p:
        p_1 = p1(p)
        p_2 = p2(p)
        ratio = (schatten_norm_jax(gradient, p=p_1) ** 2 /
                 schatten_norm_jax(A, p=p_2) ** 2)
        if ratio > max_norm:
            max_norm = ratio
            p_max = p
    return p_max, p1(p_max), p2(p_max)


In [ ]:
def ratio(gradient, A, p_1):
    # SVD on GPU (CuPy) then immediately pass numpy copy to JAX
    U, S, Vt = xp.linalg.svd(gradient, full_matrices=False)
    if p_1 == 1:
        return (schatten_norm_jax(gradient, p=p_1) ** (2 * p_1) /
                schatten_norm_jax(U @ Vt @ A, p=2) ** 2)
    else:
        return (schatten_norm_jax(gradient, p=p_1) ** (2 * p_1) /
                schatten_norm_jax(U @ xp.diag(S ** (p_1 - 1)) @ Vt @ A, p=2) ** 2)


In [ ]:
def find_p(gradient, A, list_p):
    max_norm = -xp.inf
    p_max = 0
    for p in list_p:
        r = ratio(gradient, A, p1(p))
        if r > max_norm:
            max_norm = r
            p_max = p
    return p_max, p1(p_max), p2(p_max)


In [ ]:
def learning_rate(gradient, A, n, p):
    U, S, Vt = xp.linalg.svd(gradient, full_matrices=False)
    p_1 = p1(p)
    if p_1 == 1:
        return (n * schatten_norm_jax(gradient, p=p_1) /
                xp.linalg.norm(U @ Vt @ A, 'fro') ** 2)
    else:
        return (n * schatten_norm_jax(gradient, p=p_1) ** p_1 /
                xp.linalg.norm(U @ xp.diag(S ** (p_1 - 1)) @ Vt @ A, 'fro') ** 2)


In [ ]:
def tuned_norm_optimizer(W0, A, Y, n, W_True, nb_iter, list_p):
    training_loss = [float(loss(W0, A, Y, n))]
    W = W0.copy()
    for i in range(1, nb_iter):
        gradient = 1 / n * (W @ A - Y) @ A.T
        p, p_dual, p_2dual = find_p(gradient, A, list_p)
        U, S, Vt = xp.linalg.svd(gradient, full_matrices=False)
        U_polar = U @ Vt if p_dual == 1 else U @ xp.diag(S ** (p_dual - 1)) @ Vt
        W = W - learning_rate(gradient, A, n, p) * U_polar
        training_loss.append(float(loss(W, A, Y, n)))
    return training_loss


In [ ]:
def tuned_norm(W0, A, Y, n, W_True, nb_iter, list_p):
    training_loss = [float(loss(W0, A, Y, n))]
    W = W0.copy()
    for i in range(1, nb_iter):
        gradient = 1 / n * (W @ A - Y) @ A.T
        p, p_dual, p_2dual = find_max(gradient, A, list_p)
        U, S, Vt = xp.linalg.svd(gradient, full_matrices=False)
        U_polar = U @ Vt if p_dual == 1 else U @ xp.diag(S ** (p_dual - 1)) @ Vt
        lr = n / (schatten_norm_jax(A, p_2dual) ** 2 *
                  schatten_norm_jax(gradient, p_dual) ** (p_dual - 2))
        W = W - lr * U_polar
        training_loss.append(float(loss(W, A, Y, n)))
    return training_loss


In [ ]:
def SpecGD(W0, A, Y, n, W_True, nb_iter):
    L_op = (xp.linalg.norm(A, 'fro') ** 2) / n
    training_loss = [float(loss(W0, A, Y, n))]
    nr = []
    W = W0.copy()
    for i in range(1, nb_iter):
        gradient = 1 / n * (W @ A - Y) @ A.T
        nr.append(float(xp.linalg.norm(gradient, 'nuc') ** 2 /
                        xp.linalg.norm(gradient, 'fro') ** 2))
        U, S, Vt = xp.linalg.svd(gradient, full_matrices=False)
        U_polar = U @ Vt
        W = W - xp.linalg.norm(gradient, 'nuc') / L_op * U_polar
        training_loss.append(float(loss(W, A, Y, n)))
    return training_loss, nr


In [ ]:
def GD(W0, A, Y, n, W_True, nb_iter):
    L_f = (xp.linalg.norm(A, 2) ** 2) / n
    training_loss = [float(loss(W0, A, Y, n))]
    nr = []
    W = W0.copy()
    for i in range(1, nb_iter):
        gradient = 1 / n * (W @ A - Y) @ A.T
        nr.append(float(xp.linalg.norm(gradient, 'nuc') ** 2 /
                        xp.linalg.norm(gradient, 'fro') ** 2))
        W = W - gradient / L_f
        training_loss.append(float(loss(W, A, Y, n)))
    return training_loss, nr


In [ ]:
def GD_optimal(W0, A, Y, n, W_True, nb_iter):
    training_loss = [float(loss(W0, A, Y, n))]
    W = W0.copy()
    for i in range(1, nb_iter):
        gradient = 1 / n * (W @ A - Y) @ A.T
        W = W - gradient * learning_rate(gradient, A, n, 2)
        training_loss.append(float(loss(W, A, Y, n)))
    return training_loss


In [ ]:
def SpecGD_optimal(W0, A, Y, n, W_True, nb_iter):
    training_loss = [float(loss(W0, A, Y, n))]
    W = W0.copy()
    for i in range(1, nb_iter):
        gradient = 1 / n * (W @ A - Y) @ A.T
        U, S, Vt = xp.linalg.svd(gradient, full_matrices=False)
        U_polar = U @ Vt
        W = W - learning_rate(gradient, A, n, float('inf')) * U_polar
        training_loss.append(float(loss(W, A, Y, n)))
    return training_loss


In [ ]:
relu_loss_GD, swiglu_nr_GD       = GD(W0, A, Y, 1000, W_True, 1000)
relu_loss_SpecGD, swiglu_nr_SpecGD = SpecGD(W0, A, Y, 1000, W_True, 1000)
relu_loss_tuned_optimizer          = tuned_norm_optimizer(W0, A, Y, n, W_True, 1000, list_p)
relu_GD_optimal                    = GD_optimal(W0, A, Y, n, W_True, 1000)
relu_specGD_optimal                = SpecGD_optimal(W0, A, Y, n, W_True, 1000)


In [ ]:
plt.plot(relu_loss_SpecGD,          label='SpecGD')
plt.plot(relu_loss_GD,               label='GD')
plt.plot(relu_GD_optimal,            label='GD_optimal')
plt.plot(relu_specGD_optimal,        label='SpecGD_optimal')
plt.plot(relu_loss_tuned_optimizer,  label='tuned_normed optimizer')
plt.yscale('log')
plt.title('ReLU random feature model')
plt.legend()
plt.show()


In [ ]:
def swiglu(A, B):
    """SwiGLU activation – vectorised via CuPy/NumPy."""
    A = A / (1 + xp.exp(-A))   # sigmoid gate
    return A * B


In [ ]:
m, n = 100, 1000
W_True = xp.random.randn(m, n)
W_1    = xp.random.randn(n, 50)
W_2    = xp.random.randn(n, 50)
X      = xp.random.randn(50, m)
A      = swiglu(W_1 @ X, W_2 @ X)
Y      = W_True @ A
W0     = xp.zeros((m, n))
list_p = xp.linspace(2, 20, 50).tolist()
list_p[-1] = float('inf')


In [ ]:
switglu_loss_GD, swiglu_nr_GD         = GD(W0, A, Y, 1000, W_True, 400)
switglu_loss_SpecGD, swiglu_nr_SpecGD  = SpecGD(W0, A, Y, 1000, W_True, 400)
switglu_loss_tuned_normed              = tuned_norm(W0, A, Y, n, W_True, 400, list_p)
switglu_loss_tuned_optimizer           = tuned_norm_optimizer(W0, A, Y, n, W_True, 400, list_p)
switglu_GD_optimal                     = GD_optimal(W0, A, Y, n, W_True, 400)
switglu_specGD_optimal                 = SpecGD_optimal(W0, A, Y, n, W_True, 400)


In [ ]:
plt.plot(switglu_loss_SpecGD,          label='SpecGD')
plt.plot(switglu_loss_GD,              label='GD')
plt.plot(switglu_GD_optimal,           label='GD_optimal')
plt.plot(switglu_specGD_optimal,       label='SpecGD_optimal')
plt.plot(switglu_loss_tuned_optimizer, label='tuned_normed optimizer')
plt.yscale('log')
plt.legend()
plt.show()
